In [1]:
%load_ext cython

In [62]:
%load_ext cython
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, Any, Optional, List, Tuple
import warnings
import json
import time
from pathlib import Path
import threading
import queue
import concurrent.futures
import requests
import hashlib
import pickle
import os
from dataclasses import dataclass, asdict
from abc import ABC, abstractmethod

warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("=== ENHANCED LLM-INSPIRED MOACP FRAMEWORK ===")

The cython extension is already loaded. To reload it, use:
  %reload_ext cython
=== ENHANCED LLM-INSPIRED MOACP FRAMEWORK ===


In [63]:
class InstanceSpecificConfigManager:
    """Manages configurations for different problem instances (Inspired by InstSpecHH)"""
    
    def __init__(self):
        self.instance_features = {}
        self.config_registry = {}
        self.performance_history = {}
        self.best_configs = {}
        self.subclass_configs = {}  # For instance subclasses
        
    def extract_instance_features(self, instance_file, weights_file, nbitems, num_objectives):
        """Extract key features from problem instance"""
        features = {
            'n_items': nbitems,
            'n_objectives': num_objectives,
            'instance_size': 'small' if nbitems <= 250 else 'medium' if nbitems <= 500 else 'large',
            'complexity': num_objectives * nbitems / 1000
        }
        
        # Try to extract more features from files
        try:
            with open(instance_file, 'r') as f:
                lines = f.readlines()
                if len(lines) > 0:
                    first_line = lines[0].strip()
                    if first_line:
                        parts = first_line.split()
                        if len(parts) >= 2:
                            features['n_objectives'] = int(parts[0])
                            features['n_items'] = int(parts[1])
            
            # Calculate capacity ratio
            with open(instance_file, 'r') as f:
                lines = f.readlines()
                if len(lines) > 1:
                    capacity_line = lines[1].strip()
                    if capacity_line:
                        capacity = float(capacity_line)
                        features['capacity_ratio'] = capacity / nbitems
        except:
            features['capacity_ratio'] = 0.5  # Default value
        
        # Create instance signature and subclass
        signature = f"{features['n_items']}_{features['n_objectives']}"
        features['signature'] = signature
        
        # Create subclass based on features (InstSpecHH approach)
        if features['capacity_ratio'] > 0.7:
            features['subclass'] = 'tight_capacity'
        elif features['capacity_ratio'] < 0.3:
            features['subclass'] = 'loose_capacity'
        else:
            features['subclass'] = 'medium_capacity'
            
        if features['n_objectives'] == 2:
            features['subclass'] += '_2obj'
        elif features['n_objectives'] == 3:
            features['subclass'] += '_3obj'
        else:
            features['subclass'] += '_4obj'
        
        self.instance_features[signature] = features
        return features
    
    def get_best_config_for_subclass(self, subclass):
        """Get best configuration for a specific subclass (InstSpecHH approach)"""
        if subclass in self.subclass_configs:
            return self.subclass_configs[subclass]
        
        # Default configurations based on subclass
        if 'tight_capacity' in subclass:
            return {
                'alpha': 45, 'kappa': 0.18, 'L': 7,
                'operator_strategy': ['swap', 'greedy_add', 'local_search'],
                'runtime_threshold': 10.0,
                'search_intensity': 'high'
            }
        elif 'loose_capacity' in subclass:
            return {
                'alpha': 35, 'kappa': 0.12, 'L': 5,
                'operator_strategy': ['swap', 'greedy_add', 'mutation'],
                'runtime_threshold': 7.0,
                'search_intensity': 'medium'
            }
        else:  # medium_capacity
            return {
                'alpha': 40, 'kappa': 0.15, 'L': 6,
                'operator_strategy': ['swap', 'greedy_add', 'local_search'],
                'runtime_threshold': 8.0,
                'search_intensity': 'medium'
            }
    
    def update_best_config(self, subclass, config, performance):
        """Update best configuration for a subclass"""
        if subclass not in self.subclass_configs:
            self.subclass_configs[subclass] = config.copy()
            return
        
        # Update if performance is better
        current_best = self.subclass_configs[subclass]
        if performance > self.performance_history.get(subclass, {}).get('best_performance', 0):
            self.subclass_configs[subclass] = config.copy()
            self.performance_history[subclass] = {
                'best_performance': performance,
                'config': config.copy(),
                'timestamp': time.time()
            }

# Initialize instance-specific config manager
instance_config_manager = InstanceSpecificConfigManager()

In [64]:
import json
import re
import numpy as np

class MindEvolutionLLMInterface:
    """Enhanced LLM interface with ultra-robust JSON parsing"""
    
    def __init__(self, model_path="llama3:latest", temperature=0.7, max_tokens=500):
        self.model_path = model_path
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.cache = {}
        self.call_count = 0
        self.successful_calls = 0
        self.failed_calls = 0
        self.connection_status = None
        self.exploration_history = []
        self.population_configs = []
        self._verify_connection()
        
    def _verify_connection(self):
        """Verify if LLaMA-3 is accessible"""
        try:
            response = requests.get('http://localhost:11434/api/tags', timeout=5)
            if response.status_code == 200:
                models = response.json().get('models', [])
                model_names = [model['name'] for model in models]
                
                if self.model_path in model_names:
                    self.connection_status = "ollama_connected"
                    print(f"✅ Ollama connected with {self.model_path} available")
                    return True
                else:
                    self.connection_status = "ollama_no_model"
                    return False
            else:
                self.connection_status = "ollama_failed"
        except Exception as e:
            self.connection_status = "ollama_unavailable"
        
        return False
    
    def _call_ollama(self, prompt):
        """Call local Ollama API for LLaMA-3"""
        try:
            response = requests.post(
                'http://localhost:11434/api/generate',
                json={
                    'model': self.model_path,
                    'prompt': prompt,
                    'stream': False,
                    'options': {
                        'temperature': self.temperature,
                        'num_predict': self.max_tokens,
                        'timeout': 30
                    }
                },
                timeout=120
            )
            if response.status_code == 200:
                self.successful_calls += 1
                return response.json()['response']
            else:
                self.failed_calls += 1
                return None
        except Exception as e:
            self.failed_calls += 1
            return None
    
    def _extract_json_from_response(self, response):
        """Ultra-robust JSON extraction from LLM response"""
        if not response:
            return None
            
        # Try multiple extraction methods
        json_str = None
        
        # Method 1: Look for JSON code blocks first (most reliable)
        pattern = r'```(?:json)?\s*(\{.*?\})\s*```'
        match = re.search(pattern, response, re.DOTALL)
        if match:
            json_str = match.group(1)
        
        # Method 2: Look for JSON between first { and last }
        if json_str is None:
            start_idx = response.find('{')
            end_idx = response.rfind('}') + 1
            if start_idx != -1 and end_idx != -1:
                json_str = response[start_idx:end_idx]
        
        # Method 3: Try to find any JSON-like structure
        if json_str is None:
            pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
            matches = re.findall(pattern, response, re.DOTALL)
            if matches:
                json_str = matches[0]  # Take the first match
        
        if json_str:
            # Fix common JSON issues before parsing
            json_str = self._fix_json_issues(json_str)
            
            try:
                return json.loads(json_str)
            except json.JSONDecodeError as e:
                print(f"⚠️ JSON parsing error: {e}")
                print(f"Problematic JSON: {json_str[:200]}...")
                
                # Try to extract values manually as last resort
                return self._extract_values_manually(json_str)
        
        return None
    
    def _fix_json_issues(self, json_str):
        """Fix common JSON issues"""
        # Fix escape sequences - use raw strings for replacement patterns
        json_str = re.sub(r'\\"', '"', json_str)  # Fix escaped quotes
        json_str = re.sub(r'\\\\', r'\\', json_str)  # Fix double backslashes
        
        # Fix broken string concatenations (like "medium""reasoning")
        json_str = re.sub(r'"\s*"\s*', '', json_str)  # Remove empty string concatenations
        
        # Fix missing commas between key-value pairs
        json_str = re.sub(r'"\s*}\s*"', r'","', json_str)  # Add comma between objects
        json_str = re.sub(r'"\s*\]\s*"', r'","', json_str)  # Add comma between arrays
        
        # Fix missing commas in arrays
        json_str = re.sub(r'"\s*"\s*', r'","', json_str)  # Fix array elements
        
        # Replace single quotes with double quotes for string values
        json_str = re.sub(r"'(\w+)'", r'"\1"', json_str)
        
        # Fix trailing commas
        json_str = re.sub(r',\s*}', '}', json_str)
        json_str = re.sub(r',\s*]', ']', json_str)
        
        # Fix unescaped quotes in strings
        json_str = re.sub(r':\s*"([^"]*)"([^"]*?)"', r': "\1\\"\\2\\""', json_str)
        
        # Clean up whitespace
        json_str = re.sub(r'\s+', ' ', json_str)
        
        # Fix broken key-value pairs
        json_str = re.sub(r'(\w+)\s*:\s*"', r'"\1": "', json_str)
        
        return json_str
    
    def _extract_values_manually(self, json_str):
        """Extract values manually as last resort"""
        try:
            # Extract alpha
            alpha_match = re.search(r'"alpha"\s*:\s*(\d+)', json_str)
            alpha = int(alpha_match.group(1)) if alpha_match else 40
            
            # Extract kappa
            kappa_match = re.search(r'"kappa"\s*:\s*([\d.]+)', json_str)
            kappa = float(kappa_match.group(1)) if kappa_match else 0.15
            
            # Extract L
            L_match = re.search(r'"L"\s*:\s*(\d+)', json_str)
            L = int(L_match.group(1)) if L_match else 6
            
            # Extract runtime_threshold
            runtime_match = re.search(r'"runtime_threshold"\s*:\s*([\d.]+)', json_str)
            runtime = float(runtime_match.group(1)) if runtime_match else 7.0
            
            # Extract search_intensity
            intensity_match = re.search(r'"search_intensity"\s*:\s*["\'](\w+)["\']', json_str)
            intensity = intensity_match.group(1) if intensity_match else 'medium'
            
            # Extract operator_strategy
            operators = []
            operator_match = re.search(r'"operator_strategy"\s*:\s*\[(.*?)\]', json_str, re.DOTALL)
            if operator_match:
                ops_str = operator_match.group(1)
                op_matches = re.findall(r'["\'](\w+)["\']', ops_str)
                operators = op_matches if op_matches else ['swap', 'greedy_add']
            else:
                operators = ['swap', 'greedy_add']
            
            # Extract reasoning
            reasoning_match = re.search(r'"reasoning"\s*:\s*["\']([^"\']*)["\']', json_str)
            reasoning = reasoning_match.group(1) if reasoning_match else "Extracted manually"
            
            return {
                'alpha': alpha,
                'kappa': kappa,
                'L': L,
                'operator_strategy': operators,
                'runtime_threshold': runtime,
                'search_intensity': intensity,
                'reasoning': reasoning
            }
        except Exception as e:
            print(f"⚠️ Manual extraction failed: {e}")
            return None
    
    def initialize_evolutionary_population(self, instance_features, target_performance):
        """Initialize population of configurations for evolutionary search"""
        
        if self.connection_status != "ollama_connected":
            return self._initialize_rule_based_population(instance_features, target_performance)
        
        population = []
        
        # Try to get configs from LLM, but have robust fallback
        for i in range(5):
            prompt = f"""
You are an expert in multi-objective optimization for knapsack problems.

INSTANCE FEATURES:
- Number of items: {instance_features['n_items']}
- Number of objectives: {instance_features['n_objectives']}
- Instance size: {instance_features['instance_size']}
- Complexity score: {instance_features['complexity']:.2f}
- Subclass: {instance_features['subclass']}

TARGET PERFORMANCE:
- Target hypervolume: {target_performance:,.0f}

TASK:
Generate a DIVERSE configuration for evolutionary search (Configuration {i+1}/5).
Each configuration should explore different regions of the parameter space.

PARAMETER CONSTRAINTS:
- alpha (population size): 20-60
- kappa (selection pressure): 0.03-0.25
- L (local search intensity): 2-12
- runtime_threshold: 5.0-15.0
- search_intensity: low, medium, or high

OPERATORS:
- swap: Exchange items in solution
- greedy_add: Add best feasible items
- mutation: Random modifications
- local_search: Neighborhood exploration

IMPORTANT: Return ONLY valid JSON. No explanations outside the JSON.

RESPONSE FORMAT:
{{
    "alpha": 40,
    "kappa": 0.150,
    "L": 6,
    "operator_strategy": ["swap", "greedy_add"],
    "runtime_threshold": 7.0,
    "search_intensity": "medium",
    "reasoning": "Brief explanation"
}}
"""
            
            response = self._call_ollama(prompt)
            
            if response:
                config = self._extract_json_from_response(response)
                
                if config:
                    # Validate and adjust parameters
                    config['alpha'] = min(60, max(20, int(config.get('alpha', 40))))
                    config['kappa'] = min(0.25, max(0.03, float(config.get('kappa', 0.15))))
                    config['L'] = min(12, max(2, int(config.get('L', 6))))
                    config['runtime_threshold'] = min(15.0, max(5.0, float(config.get('runtime_threshold', 7.0))))
                    
                    # Validate operators
                    valid_operators = ['swap', 'greedy_add', 'mutation', 'local_search']
                    strategy = config.get('operator_strategy', ['swap', 'greedy_add'])
                    if isinstance(strategy, str):
                        strategy = [strategy]
                    config['operator_strategy'] = [op for op in strategy if op in valid_operators]
                    
                    if not config['operator_strategy']:
                        config['operator_strategy'] = ['swap', 'greedy_add', 'mutation']
                    
                    # Validate search intensity
                    if config.get('search_intensity') not in ['low', 'medium', 'high']:
                        config['search_intensity'] = 'medium'
                    
                    population.append(config)
                    print(f"✅ Successfully generated config {i+1} from LLM")
                else:
                    print(f"⚠️ Could not parse JSON from LLM response for config {i+1}")
                    # Add fallback config
                    population.append(self._generate_single_rule_based_config(instance_features, target_performance))
            else:
                print(f"⚠️ No response from LLM for config {i+1}")
                # Add fallback config
                population.append(self._generate_single_rule_based_config(instance_features, target_performance))
        
        # Ensure we have at least 5 configs
        while len(population) < 5:
            population.append(self._generate_single_rule_based_config(instance_features, target_performance))
        
        self.population_configs = population
        return population
    
    def _generate_single_rule_based_config(self, instance_features, target_performance):
        """Generate a single rule-based configuration"""
        if instance_features['instance_size'] == 'small':
            alpha = np.random.choice([30, 35, 40])
            kappa = np.random.choice([0.08, 0.10, 0.12])
            L = np.random.choice([4, 5, 6])
            runtime = 7.0
        elif instance_features['instance_size'] == 'medium':
            alpha = np.random.choice([40, 45, 50])
            kappa = np.random.choice([0.12, 0.15, 0.18])
            L = np.random.choice([6, 7, 8])
            runtime = 10.0
        else:
            alpha = np.random.choice([45, 50, 55])
            kappa = np.random.choice([0.15, 0.18, 0.20])
            L = np.random.choice([7, 8, 9])
            runtime = 12.0
        
        if instance_features['n_objectives'] == 2:
            operator_strategy = ['swap', 'greedy_add', 'mutation']
        else:
            operator_strategy = ['swap', 'greedy_add', 'local_search']
        
        return {
            'alpha': alpha,
            'kappa': kappa,
            'L': L,
            'operator_strategy': operator_strategy,
            'runtime_threshold': runtime,
            'search_intensity': 'medium',
            'reasoning': 'Rule-based config'
        }
    
    def _initialize_rule_based_population(self, instance_features, target_performance):
        """Initialize population with rule-based configurations"""
        population = []
        
        # Generate diverse configurations
        base_configs = [
            {'alpha': 30, 'kappa': 0.10, 'L': 4, 'search_intensity': 'low'},
            {'alpha': 40, 'kappa': 0.15, 'L': 6, 'search_intensity': 'medium'},
            {'alpha': 50, 'kappa': 0.20, 'L': 8, 'search_intensity': 'high'},
            {'alpha': 35, 'kappa': 0.12, 'L': 5, 'search_intensity': 'medium'},
            {'alpha': 45, 'kappa': 0.18, 'L': 7, 'search_intensity': 'high'}
        ]
        
        for base_config in base_configs:
            config = base_config.copy()
            
            # Set operators based on instance type
            if instance_features['n_objectives'] == 2:
                if config['search_intensity'] == 'high':
                    config['operator_strategy'] = ['swap', 'greedy_add', 'local_search']
                else:
                    config['operator_strategy'] = ['swap', 'greedy_add', 'mutation']
            else:
                config['operator_strategy'] = ['swap', 'greedy_add', 'local_search']
            
            # Set runtime threshold
            if instance_features['instance_size'] == 'small':
                config['runtime_threshold'] = 7.0
            elif instance_features['instance_size'] == 'medium':
                config['runtime_threshold'] = 10.0
            else:
                config['runtime_threshold'] = 12.0
            
            config['reasoning'] = f"Rule-based config with {config['search_intensity']} intensity"
            population.append(config)
        
        self.population_configs = population
        return population
    
    def evolutionary_crossover(self, parent1, parent2, instance_features):
        """Perform crossover between two parent configurations"""
        
        if self.connection_status != "ollama_connected":
            return self._rule_based_crossover(parent1, parent2, instance_features)
        
        prompt = f"""
You are performing evolutionary crossover for multi-objective optimization.

INSTANCE FEATURES:
- Number of items: {instance_features['n_items']}
- Number of objectives: {instance_features['n_objectives']}
- Instance size: {instance_features['instance_size']}
- Subclass: {instance_features['subclass']}

PARENT CONFIGURATIONS:
Parent 1: {parent1}
Parent 2: {parent2}

TASK:
Create a CHILD configuration by combining the best aspects of both parents.
The child should inherit beneficial traits from both parents while maintaining diversity.

PARAMETER CONSTRAINTS:
- alpha (population size): 20-60
- kappa (selection pressure): 0.03-0.25
- L (local search intensity): 2-12
- runtime_threshold: 5.0-15.0
- search_intensity: low, medium, or high

IMPORTANT: Return ONLY valid JSON. No explanations outside the JSON.

RESPONSE FORMAT:
{{
    "alpha": 40,
    "kappa": 0.150,
    "L": 6,
    "operator_strategy": ["swap", "greedy_add"],
    "runtime_threshold": 7.0,
    "search_intensity": "medium",
    "reasoning": "Brief explanation"
}}
"""
        
        response = self._call_ollama(prompt)
        
        if response:
            child = self._extract_json_from_response(response)
            
            if child:
                # Validate and adjust parameters
                child['alpha'] = min(60, max(20, int(child.get('alpha', 40))))
                child['kappa'] = min(0.25, max(0.03, float(child.get('kappa', 0.15))))
                child['L'] = min(12, max(2, int(child.get('L', 6))))
                child['runtime_threshold'] = min(15.0, max(5.0, float(child.get('runtime_threshold', 7.0))))
                
                # Validate operators
                valid_operators = ['swap', 'greedy_add', 'mutation', 'local_search']
                strategy = child.get('operator_strategy', ['swap', 'greedy_add'])
                if isinstance(strategy, str):
                    strategy = [strategy]
                child['operator_strategy'] = [op for op in strategy if op in valid_operators]
                
                if not child['operator_strategy']:
                    child['operator_strategy'] = ['swap', 'greedy_add', 'mutation']
                
                # Validate search intensity
                if child.get('search_intensity') not in ['low', 'medium', 'high']:
                    child['search_intensity'] = 'medium'
                
                return child
        
        # Fallback to rule-based crossover
        return self._rule_based_crossover(parent1, parent2, instance_features)
    
    def _rule_based_crossover(self, parent1, parent2, instance_features):
        """Perform rule-based crossover between two parent configurations"""
        
        child = {}
        
        # Crossover alpha (average of parents)
        child['alpha'] = int((parent1['alpha'] + parent2['alpha']) / 2)
        
        # Crossover kappa (average of parents)
        child['kappa'] = (parent1['kappa'] + parent2['kappa']) / 2
        
        # Crossover L (average of parents)
        child['L'] = int((parent1['L'] + parent2['L']) / 2)
        
        # Crossover operators (union of parent operators)
        operators1 = set(parent1['operator_strategy'])
        operators2 = set(parent2['operator_strategy'])
        child['operator_strategy'] = list(operators1.union(operators2))[:3]  # Limit to 3 operators
        
        if len(child['operator_strategy']) < 2:
            child['operator_strategy'] = ['swap', 'greedy_add', 'mutation']
        
        # Crossover runtime threshold (average of parents)
        child['runtime_threshold'] = (parent1['runtime_threshold'] + parent2['runtime_threshold']) / 2
        
        # Crossover search intensity (choose from parents)
        child['search_intensity'] = parent1['search_intensity'] if np.random.random() > 0.5 else parent2['search_intensity']
        
        child['reasoning'] = f"Rule-based crossover of parents"
        
        return child
    
    def evolutionary_mutation(self, config, instance_features, mutation_strength='medium'):
        """Perform mutation on a configuration"""
        
        if self.connection_status != "ollama_connected":
            return self._rule_based_mutation(config, instance_features, mutation_strength)
        
        prompt = f"""
You are performing evolutionary mutation for multi-objective optimization.

INSTANCE FEATURES:
- Number of items: {instance_features['n_items']}
- Number of objectives: {instance_features['n_objectives']}
- Instance size: {instance_features['instance_size']}
- Subclass: {instance_features['subclass']}

CURRENT CONFIGURATION:
{config}

MUTATION STRENGTH: {mutation_strength}

TASK:
Create a MUTATED configuration by introducing controlled random changes.
The mutation should explore new regions while maintaining the core strengths.

PARAMETER CONSTRAINTS:
- alpha (population size): 20-60
- kappa (selection pressure): 0.03-0.25
- L (local search intensity): 2-12
- runtime_threshold: 5.0-15.0
- search_intensity: low, medium, or high

IMPORTANT: Return ONLY valid JSON. No explanations outside the JSON.

RESPONSE FORMAT:
{{
    "alpha": 40,
    "kappa": 0.150,
    "L": 6,
    "operator_strategy": ["swap", "greedy_add"],
    "runtime_threshold": 7.0,
    "search_intensity": "medium",
    "reasoning": "Brief explanation"
}}
"""
        
        response = self._call_ollama(prompt)
        
        if response:
            mutated = self._extract_json_from_response(response)
            
            if mutated:
                # Validate and adjust parameters
                mutated['alpha'] = min(60, max(20, int(mutated.get('alpha', 40))))
                mutated['kappa'] = min(0.25, max(0.03, float(mutated.get('kappa', 0.15))))
                mutated['L'] = min(12, max(2, int(mutated.get('L', 6))))
                mutated['runtime_threshold'] = min(15.0, max(5.0, float(mutated.get('runtime_threshold', 7.0))))
                
                # Validate operators
                valid_operators = ['swap', 'greedy_add', 'mutation', 'local_search']
                strategy = mutated.get('operator_strategy', ['swap', 'greedy_add'])
                if isinstance(strategy, str):
                    strategy = [strategy]
                mutated['operator_strategy'] = [op for op in strategy if op in valid_operators]
                
                if not mutated['operator_strategy']:
                    mutated['operator_strategy'] = ['swap', 'greedy_add', 'mutation']
                
                # Validate search intensity
                if mutated.get('search_intensity') not in ['low', 'medium', 'high']:
                    mutated['search_intensity'] = 'medium'
                
                return mutated
        
        # Fallback to rule-based mutation
        return self._rule_based_mutation(config, instance_features, mutation_strength)
    
    def _rule_based_mutation(self, config, instance_features, mutation_strength='medium'):
        """Perform rule-based mutation on a configuration"""
        
        mutated = config.copy()
        
        # Determine mutation magnitude based on strength
        if mutation_strength == 'low':
            alpha_mutation = np.random.choice([-3, 0, 3])
            kappa_mutation = np.random.choice([-0.02, 0, 0.02])
            L_mutation = np.random.choice([-1, 0, 1])
        elif mutation_strength == 'high':
            alpha_mutation = np.random.choice([-8, -5, 0, 5, 8])
            kappa_mutation = np.random.choice([-0.05, -0.03, 0, 0.03, 0.05])
            L_mutation = np.random.choice([-2, -1, 0, 1, 2])
        else:  # medium
            alpha_mutation = np.random.choice([-5, -3, 0, 3, 5])
            kappa_mutation = np.random.choice([-0.03, -0.02, 0, 0.02, 0.03])
            L_mutation = np.random.choice([-1, 0, 1])
        
        # Apply mutations
        mutated['alpha'] = min(60, max(20, config['alpha'] + alpha_mutation))
        mutated['kappa'] = min(0.25, max(0.03, config['kappa'] + kappa_mutation))
        mutated['L'] = min(12, max(2, config['L'] + L_mutation))
        
        # Mutate operators with some probability
        if np.random.random() < 0.3:
            valid_operators = ['swap', 'greedy_add', 'mutation', 'local_search']
            current_ops = set(config['operator_strategy'])
            
            # Remove one operator and add another
            if len(current_ops) > 2:
                removed = np.random.choice(list(current_ops))
                current_ops.remove(removed)
            
            available_ops = [op for op in valid_operators if op not in current_ops]
            if available_ops:
                added = np.random.choice(available_ops)
                current_ops.add(added)
            
            mutated['operator_strategy'] = list(current_ops)
        
        # Mutate search intensity with some probability
        if np.random.random() < 0.2:
            intensities = ['low', 'medium', 'high']
            current_idx = intensities.index(config['search_intensity'])
            new_idx = (current_idx + np.random.choice([-1, 1])) % 3
            mutated['search_intensity'] = intensities[new_idx]
        
        mutated['reasoning'] = f"Rule-based {mutation_strength} mutation"
        
        return mutated

In [65]:
# Debug LLM responses
def debug_llm_response():
    """Debug function to test LLM response parsing"""
    llm = MindEvolutionLLMInterface(model_path="llama3:latest", temperature=0.7)
    
    test_prompt = """
Generate a simple configuration in JSON format:
{
    "alpha": 40,
    "kappa": 0.15,
    "L": 6,
    "operator_strategy": ["swap", "greedy_add"],
    "runtime_threshold": 7.0,
    "search_intensity": "medium",
    "reasoning": "test configuration"
}
"""
    
    response = llm._call_ollama(test_prompt)
    print("Raw LLM response:")
    print(response)
    print("\n" + "="*50)
    
    parsed = llm._extract_json_from_response(response)
    print("Parsed JSON:")
    print(parsed)

# Run debug if needed
debug_llm_response()

✅ Ollama connected with llama3:latest available
Raw LLM response:
Here is the simple configuration in JSON format:

```
{
    "alpha": 40,
    "kappa": 0.15,
    "L": 6,
    "operator_strategy": ["swap", "greedy_add"],
    "runtime_threshold": 7.0,
    "search_intensity": "medium",
    "reasoning": "test configuration"
}
```

Let me know if you need anything else!

⚠️ JSON parsing error: Invalid \escape: line 1 column 139 (char 138)
Problematic JSON: { "alpha": 40, "kappa": 0.15, "L": 6, "operator_strategy": ["swap", "greedy_add"], "runtime_threshold": 7.0, "search_intensity": "medium\"\2\""reasoning": "test configuration" }...
Parsed JSON:
{'alpha': 40, 'kappa': 0.15, 'L': 6, 'operator_strategy': ['swap', 'greedy_add'], 'runtime_threshold': 7.0, 'search_intensity': 'medium', 'reasoning': 'test configuration'}


In [66]:
%%cython
"""
Enhanced MOACP Implementation with Mind Evolution and Instance-Specific Optimization
"""

from libc.stdlib cimport malloc, free, srand, rand
from libc.string cimport memset
from libc.math cimport exp
import numpy as np
import time

# Structs
cdef struct ind:
    int nombr_nonpris
    int nombr
    int rank
    float fitnessbest
    float fitness
    int explored
    double *f
    double *capa
    double *v
    int *d
    int *Items

cdef struct pop:
    int size
    int maxsize
    ind **ind_array

# Enhanced agent struct for Mind Evolution
cdef struct agent:
    int agent_id
    double performance_score
    int generation
    int parent1_id
    int parent2_id
    bint is_mutant
    double *config_params

# Globals
cdef int NBITEMS = 250
cdef int ni = 250
cdef int L = 5
cdef double LARGE = 10e50
cdef float smallValue = 0.0000001
cdef double kappa = 0.05
cdef int alpha = 10
cdef int paretoIni = 28000

cdef int nf = 2
cdef double *capacities = NULL
cdef int **weights = NULL
cdef int **profits = NULL
cdef double *vector_weight = NULL
cdef double max_bound = 0.0
cdef double **OBJ_Weights = NULL
cdef int nombreLIGNE = 0
cdef int nextLn = 0
cdef int inv = 0
cdef int OBJ_Weights_lines = 0

# Agent population for Mind Evolution
cdef agent *agent_population = NULL
cdef int num_agents = 5

def seed(int x):
    srand(x)

cdef int irand(int range_val):
    return rand() % range_val

cdef void *chk_malloc(size_t size):
    cdef void *return_value = malloc(size)
    if return_value == NULL:
        raise MemoryError("Out of memory.")
    memset(return_value, 0, size)
    return return_value

cdef pop *create_pop(int maxsize, int nf):
    cdef int i
    cdef pop *pp = <pop *>chk_malloc(sizeof(pop))
    pp.size = 0
    pp.maxsize = maxsize
    pp.ind_array = <ind **>chk_malloc(maxsize * sizeof(void*))
    for i in range(maxsize):
        pp.ind_array[i] = NULL
    return pp

cdef ind *create_ind(int nf):
    cdef int i
    cdef ind *p_ind = <ind *>chk_malloc(sizeof(ind))
    p_ind.nombr_nonpris = 0
    p_ind.nombr = 0
    p_ind.rank = 0
    p_ind.fitnessbest = -1.0
    p_ind.fitness = -1.0
    p_ind.explored = 0
    p_ind.f = <double *>chk_malloc(nf * sizeof(double))
    p_ind.capa = <double *>chk_malloc(nf * sizeof(double))
    p_ind.v = <double *>chk_malloc(nf * sizeof(double))
    p_ind.d = <int *>chk_malloc(ni * sizeof(int))
    p_ind.Items = <int *>chk_malloc(ni * sizeof(int))
    for i in range(ni):
        p_ind.Items[i] = 0
        p_ind.d[i] = 0
    for i in range(nf):
        p_ind.f[i] = 0.0
        p_ind.capa[i] = 0.0
        p_ind.v[i] = 0.0
    return p_ind

cdef ind *ind_copy(ind *i):
    cdef ind *p_ind = create_ind(nf)
    cdef int k
    p_ind.nombr_nonpris = i.nombr_nonpris
    p_ind.nombr = i.nombr
    p_ind.rank = i.rank
    p_ind.fitnessbest = i.fitnessbest
    p_ind.fitness = i.fitness
    p_ind.explored = i.explored
    for k in range(nf):
        p_ind.f[k] = i.f[k]
        p_ind.v[k] = i.v[k]
        p_ind.capa[k] = i.capa[k]
    for k in range(ni):
        p_ind.d[k] = i.d[k]
        p_ind.Items[k] = i.Items[k]
    return p_ind

cdef void free_ind(ind *p_ind):
    if p_ind != NULL:
        free(p_ind.d)
        free(p_ind.f)
        free(p_ind.capa)
        free(p_ind.v)
        free(p_ind.Items)
        free(p_ind)

cdef void complete_free_pop(pop *pp):
    cdef int i
    if pp != NULL:
        if pp.ind_array != NULL:
            for i in range(pp.size):
                if pp.ind_array[i] != NULL:
                    free_ind(pp.ind_array[i])
                    pp.ind_array[i] = NULL
            free(pp.ind_array)
        free(pp)

cdef void cleanup_globals():
    global capacities, weights, profits, vector_weight, OBJ_Weights, OBJ_Weights_lines, nf, ni
    if capacities != NULL:
        free(capacities)
        capacities = NULL
    if weights != NULL:
        for i in range(nf):
            if weights[i] != NULL:
                free(weights[i])
        free(weights)
        weights = NULL
    if profits != NULL:
        for i in range(nf):
            if profits[i] != NULL:
                free(profits[i])
        free(profits)
        profits = NULL
    if vector_weight != NULL:
        free(vector_weight)
        vector_weight = NULL
    if OBJ_Weights != NULL:
        for i in range(nf):
            if OBJ_Weights[i] != NULL:
                free(OBJ_Weights[i])
        free(OBJ_Weights)
        OBJ_Weights = NULL
    OBJ_Weights_lines = 0
    nf = 0
    ni = 0

cdef int non_dominated(ind *p_ind_a, ind *p_ind_b):
    cdef int i
    cdef int a_is_good = -1
    cdef int equal = 1
    for i in range(nf):
        if p_ind_a.f[i] > p_ind_b.f[i]:
            a_is_good = 1
        if p_ind_a.f[i] != p_ind_b.f[i]:
            equal = 0
    if equal:
        return 0
    return a_is_good

cdef double calcAddEpsIndicator(ind *p_ind_a, ind *p_ind_b):
    global max_bound
    cdef int i
    cdef double eps
    cdef double temp_eps
    if max_bound == 0.0:
        max_bound = 1e-8
    eps = (p_ind_a.v[0]/max_bound)-(p_ind_b.v[0]/max_bound)
    for i in range(1, nf):
        temp_eps = (p_ind_a.v[i]/max_bound)-(p_ind_b.v[i]/max_bound)
        if temp_eps > eps:
            eps = temp_eps
    return eps

cdef void init_fitness(ind *x):
    x.fitness = 0.0

cdef void update_fitness(ind *x, double I):
    x.fitness -= exp(-I / kappa)

cdef double update_fitness_return(double f, double I):
    return f - exp(-I / kappa)

cdef int delete_fitness(ind *x, double I):
    x.fitness += exp(-I / kappa)
    return 0

cdef void compute_ind_fitness(ind *x, pop *SP):
    cdef int j
    init_fitness(x)
    for j in range(SP.size):
        if SP.ind_array[j] != x:
            update_fitness(x, calcAddEpsIndicator(SP.ind_array[j], x))

cdef void compute_all_fitness(pop *SP):
    cdef int i
    for i in range(SP.size):
        compute_ind_fitness(SP.ind_array[i], SP)

cdef void loadMOKP(char *filename):
    global nf, ni, capacities, weights, profits
    cdef int i, f
    with open(filename.decode(), "r") as source:
        _nf, _ni = [int(x) for x in source.readline().split()]
        nf = _nf
        ni = _ni
        capacities = <double *>chk_malloc(nf * sizeof(double))
        weights = <int **>chk_malloc(nf * sizeof(void*))
        profits = <int **>chk_malloc(nf * sizeof(void*))
        for f in range(nf):
            capacities[f] = float(source.readline().strip())
            weights[f] = <int *>chk_malloc(ni * sizeof(int))
            profits[f] = <int *>chk_malloc(ni * sizeof(int))
            for i in range(ni):
                source.readline()  # item index (ignore)
                weights[f][i] = int(source.readline().strip())
                profits[f][i] = int(source.readline().strip())

cdef void read_weights_file(char *filename):
    global OBJ_Weights, nombreLIGNE, nf, OBJ_Weights_lines
    cdef int i, j, nlines
    with open(filename.decode(), "r") as f:
        lines = [line for line in f if line.strip()]
    nlines = len(lines)
    OBJ_Weights = <double **>chk_malloc(nf * sizeof(void*))
    for i in range(nf):
        OBJ_Weights[i] = <double *>chk_malloc(nlines * sizeof(double))
    for i, line in enumerate(lines):
        vals = line.strip().split()
        for j in range(nf):
            OBJ_Weights[j][i] = float(vals[j])
    nombreLIGNE = nlines - 1
    OBJ_Weights_lines = nlines

cdef void dynamic_weight_allpop():
    global vector_weight, OBJ_Weights, nombreLIGNE, nf, nextLn
    cdef int i
    if vector_weight == NULL:
        vector_weight = <double *>chk_malloc(nf * sizeof(double))
    for i in range(nf):
        vector_weight[i] = OBJ_Weights[i][nextLn]
    if nextLn == nombreLIGNE:
        nextLn = 0
    else:
        nextLn += 1

cdef void choose_weight():
    dynamic_weight_allpop()

cdef void random_init_ind(ind *x):
    cdef int j, r, tmp
    for j in range(ni):
        x.d[j] = j
    for j in range(ni):
        r = irand(ni)
        tmp = x.d[r]
        x.d[r] = x.d[j]
        x.d[j] = tmp

cdef void evaluate(ind *x):
    cdef int j, l, k, faisable
    x.nombr = 0
    x.nombr_nonpris = 0
    for j in range(nf):
        x.capa[j] = 0.0
        x.f[j] = 0.0
    for j in range(ni):
        l = 0
        faisable = 1
        while l < nf and faisable == 1:
            if x.capa[l] + weights[l][x.d[j]] > capacities[l]:
                faisable = 0
            l += 1
        if faisable == 1:
            for k in range(nf):
                x.capa[k] += weights[k][x.d[j]]
                x.f[k] += profits[k][x.d[j]]
            x.Items[x.d[j]] = 1
            x.nombr += 1
        else:
            x.Items[x.d[j]] = 0
            x.nombr_nonpris += 1

cdef void P_init_pop(pop *SP, pop *Sarchive, int alpha):
    cdef int i, x, tmp, t
    t = max(alpha, Sarchive.size)
    cdef int* shuffle = <int *>chk_malloc(t * sizeof(int))
    for i in range(t):
        shuffle[i] = i
    for i in range(t):
        x = irand(alpha)
        tmp = shuffle[i]
        shuffle[i] = shuffle[x]
        shuffle[x] = tmp
    SP.size = alpha
    if Sarchive.size > alpha:
        for i in range(alpha):
            SP.ind_array[i] = ind_copy(Sarchive.ind_array[shuffle[i]])
    else:
        for i in range(alpha):
            if shuffle[i] < Sarchive.size:
                SP.ind_array[i] = ind_copy(Sarchive.ind_array[shuffle[i]])
            else:
                SP.ind_array[i] = create_ind(nf)
                random_init_ind(SP.ind_array[i])
                evaluate(SP.ind_array[i])
    free(shuffle)

cdef int extractPtoArchive(pop *P, pop *archive):
    cdef int i, j, dom, t, convergence_rate
    t = archive.size + P.size
    archiveAndP = create_pop(t, nf)
    convergence_rate = 0
    for i in range(archive.size):
        archiveAndP.ind_array[i] = archive.ind_array[i]
    for i in range(P.size):
        archiveAndP.ind_array[i + archive.size] = ind_copy(P.ind_array[i])
    archiveAndP.size = t
    archive.size = 0
    for i in range(t):
        for j in range(t):
            if i != j:
                dom = non_dominated(archiveAndP.ind_array[i], archiveAndP.ind_array[j])
                if dom == -1 or (dom == 0 and i > j):
                    break
        else:
            archive.ind_array[archive.size] = ind_copy(archiveAndP.ind_array[i])
            archive.size += 1
            if i >= t - P.size:
                convergence_rate += 1
    complete_free_pop(archiveAndP)
    return convergence_rate

cdef double calcMaxbound(pop *SP, int size):
    global max_bound
    cdef int i, j
    SP.size = size
    cdef double max_b = SP.ind_array[0].v[0]
    for i in range(SP.size):
        for j in range(nf):
            if max_b < SP.ind_array[i].v[j]:
                max_b = SP.ind_array[i].v[j]
    if max_b == 0.0:
        max_b = 1e-8
    max_bound = max_b
    return max_b

cdef void calcul_weight(pop *SP, int size):
    cdef int i, j
    for i in range(SP.size):
        for j in range(nf):
            SP.ind_array[i].v[j] = SP.ind_array[i].f[j] * vector_weight[j]

cdef int compute_fitness_and_select(pop *SP, ind *x, int size):
    cdef int i, worst
    cdef double worst_fit, fit_tmp
    SP.size = size
    x.fitness = 0
    compute_ind_fitness(x, SP)
    worst_fit = x.fitness
    worst = -1
    for i in range(SP.size):
        fit_tmp = update_fitness_return(SP.ind_array[i].fitness, calcAddEpsIndicator(x, SP.ind_array[i]))
        if fit_tmp > worst_fit:
            worst = i
            worst_fit = fit_tmp
    fit_tmp = x.fitness
    if worst == -1:
        return -1
    else:
        for i in range(SP.size):
            delete_fitness(SP.ind_array[i], calcAddEpsIndicator(SP.ind_array[worst], SP.ind_array[i]))
            update_fitness(SP.ind_array[i], calcAddEpsIndicator(x, SP.ind_array[i]))
        delete_fitness(x, calcAddEpsIndicator(SP.ind_array[worst], x))
        free_ind(SP.ind_array[worst])
        SP.ind_array[worst] = ind_copy(x)
        if fit_tmp - worst_fit > smallValue:
            return worst
        else:
            return -1

# Enhanced operators with Mind Evolution inspiration
cdef void mind_evolution_operator(pop *SP, pop *Sarchive, int size, double time_budget, int generation):
    cdef ind *x
    cdef int i, j, r, t, k, l, v, sol, mino, mp, maxp, consistant, pos, stop, convergence, ii, tmp_pris, tmp_nonpris, taille, feasible, tv, IM
    cdef int* remplace = <int *>chk_malloc(L * sizeof(int))
    cdef double start_time = time.time()
    cdef double current_time
    
    SP.size = size
    extractPtoArchive(SP, Sarchive)
    
    # Adapt behavior based on generation (early generations explore more)
    cdef double exploration_rate = 0.8 - (generation * 0.05)  # Decrease exploration over time
    if exploration_rate < 0.3:
        exploration_rate = 0.3
    
    cdef int no_improvement_count = 0
    cdef double best_hv = 0.0
    
    while (time.time() - start_time) < time_budget * 0.9 and no_improvement_count < 3:
        convergence = 0
        for i in range(SP.size):
            current_time = time.time()
            
            if (current_time - start_time) > time_budget * 0.9:
                break
                
            if not SP.ind_array[i].explored:
                x = ind_copy(SP.ind_array[i])
                j = 0
                while j < x.nombr and (current_time - start_time) < time_budget * 0.9:
                    for l in range(L):
                        remplace[l] = 0
                    
                    # Mind Evolution: exploration vs exploitation
                    if irand(100) < (exploration_rate * 100):
                        # Exploration: try more diverse changes
                        mino = irand(ni)
                        while x.Items[mino] != 1 and j < x.nombr:
                            mino = irand(ni)
                    else:
                        # Exploitation: focus on promising areas
                        mino = irand(min(ni, x.nombr))
                        count = 0
                        while x.Items[mino] != 1 and count < x.nombr:
                            mino = (mino + 1) % ni
                            count += 1
                    
                    if x.Items[mino] == 1:
                        x.Items[mino] = 0
                        x.nombr -= 1
                        x.nombr_nonpris += 1
                        for r in range(nf):
                            x.capa[r] -= weights[r][mino]
                            x.f[r] -= profits[r][mino]
                        
                        IM = 0
                        taille = 0
                        while IM < L and (current_time - start_time) < time_budget * 0.9:
                            while True:
                                maxp = irand(ni)
                                if x.Items[maxp] == 0:
                                    break
                            if maxp != mino:
                                consistant = 1
                                r = 0
                                while r < nf and consistant == 1:
                                    if x.capa[r] + weights[r][maxp] > capacities[r]:
                                        consistant = 0
                                    r += 1
                                if consistant == 1:
                                    feasible = 1
                                    r = 0
                                    while r < taille and feasible:
                                        if maxp == remplace[r]:
                                            feasible = 0
                                        r += 1
                                    if feasible == 1:
                                        remplace[taille] = maxp
                                        taille += 1
                                        x.Items[maxp] = 1
                                        x.nombr_nonpris -= 1
                                        x.nombr += 1
                                        for r in range(nf):
                                            x.capa[r] += weights[r][maxp]
                                            x.f[r] += profits[r][maxp]
                            IM += 1
                        
                        for tv in range(nf):
                            x.v[tv] = x.f[tv] * vector_weight[tv]
                        max_bound = calcMaxbound(SP, SP.size)
                        sol = compute_fitness_and_select(SP, x, SP.size)
                        if sol != -1:
                            j = x.nombr + 1
                            if sol > i and i + 1 < SP.size:
                                y = SP.ind_array[i + 1]
                                SP.ind_array[i + 1] = SP.ind_array[sol]
                                SP.ind_array[sol] = y
                                i += 1
                            break
                        elif sol == -1:
                            x.Items[mino] = 1
                            x.nombr_nonpris -= 1
                            x.nombr += 1
                            for r in range(nf):
                                x.capa[r] += weights[r][mino]
                                x.f[r] += profits[r][mino]
                            if taille >= 1:
                                for r in range(taille):
                                    x.Items[remplace[r]] = 0
                                    x.nombr -= 1
                                    x.nombr_nonpris += 1
                                    for t in range(nf):
                                        x.capa[t] -= weights[t][remplace[r]]
                                        x.f[t] -= profits[t][remplace[r]]
                                        x.v[t] = x.f[t] * vector_weight[t]
                    j += 1
                tmp_pris = x.nombr
                tmp_nonpris = x.nombr_nonpris
                free_ind(x)
                if j == tmp_pris:
                    SP.ind_array[i].explored = 1
        convergence = extractPtoArchive(SP, Sarchive)
        if not convergence:
            no_improvement_count += 1
        else:
            no_improvement_count = 0
    free(remplace)

# Enhanced parameter configuration with agent influence
cdef void set_enhanced_parameters(
    object custom_params=None, 
    object operator_strategy=None, 
    bint print_params=True,
    int agent_id=-1
):
    global alpha, kappa, L, smallValue
    
    if custom_params is not None:
        # Apply enhanced bounds with validation
        new_alpha = int(custom_params.get('alpha', 40))
        new_kappa = float(custom_params.get('kappa', 0.15))
        new_L = int(custom_params.get('L', 6))
        new_small_value = float(custom_params.get('small_value', 1e-7))
        
        # Validate bounds
        alpha = min(60, max(20, new_alpha))
        kappa = min(0.25, max(0.03, new_kappa))
        L = min(12, max(2, new_L))
        smallValue = min(1e-6, max(1e-9, new_small_value))
        
        if print_params:
            agent_info = f" (Agent {agent_id})" if agent_id >= 0 else ""
            print(f"🚀 Enhanced Parameters{agent_info}: alpha={alpha}, kappa={kappa:.3f}, L={L}, small={smallValue:.1e}")
    else:
        alpha = 40
        kappa = 0.15
        L = 6
        smallValue = 1e-7
        if print_params:
            print(f"📊 Default Parameters: alpha={alpha}, kappa={kappa:.3f}, L={L}, small={smallValue:.1e}")
    
    if operator_strategy is not None:
        print(f"   Operator strategy: {operator_strategy}")

# Enhanced main MOACP runner with Mind Evolution
def run_moacp_mind_evolution(
    instance_file,
    weights_file,
    nbitems,
    num_objectives,
    custom_params=None,
    operator_strategy=None,
    print_params=True,
    runtime_threshold=7.0,
    num_runs=8,
    num_iterations=100,
    agent_id=-1,
    generation=0
):
    """
    Enhanced MOACP runner with Mind Evolution and instance-specific optimization
    """
    global nf, ni, NBITEMS, alpha, paretoIni, L, nombreLIGNE, nextLn, inv, vector_weight
    global capacities, weights, profits, OBJ_Weights

    set_enhanced_parameters(custom_params, operator_strategy, print_params, agent_id)
    NBITEMS = nbitems
    ni = nbitems
    nf = num_objectives
    paretoIni = 28000

    all_pareto_solutions = []
    run_times = []

    if print_params:
        gen_info = f" (Gen {generation})" if generation > 0 else ""
        agent_info = f" (Agent {agent_id})" if agent_id >= 0 else ""
        print(f"\n Enhanced MOACP{agent_info}{gen_info}: {num_runs} runs × {num_iterations} iterations")
        print(f" Runtime threshold: {runtime_threshold:.2f}s per run")

    total_start_time = time.time()

    for run in range(1, num_runs + 1):
        run_start_time = time.time()
        if print_params:
            print(f"   Run {run}/{num_runs}...", end=" ")

        nombreLIGNE = 0
        nextLn = 0
        inv = 0

        seed(run + agent_id * 1000)  # Unique seed per agent
        loadMOKP(instance_file.encode())
        read_weights_file(weights_file.encode())

        vector_weight = <double *>chk_malloc(nf * sizeof(double))
        P = create_pop(paretoIni, nf)

        it = 0
        while it < num_iterations:
            iteration_start = time.time()
            
            solutions = create_pop(alpha, nf)
            archive = create_pop(paretoIni, nf)
            choose_weight()
            P_init_pop(solutions, P, alpha)
            extractPtoArchive(solutions, P)
            calcul_weight(solutions, alpha)
            calcMaxbound(solutions, alpha)
            compute_all_fitness(solutions)

            # Use Mind Evolution operator
            mind_evolution_operator(solutions, archive, alpha, runtime_threshold * 0.6, generation)

            extractPtoArchive(archive, P)
            it += 1
            complete_free_pop(solutions)
            complete_free_pop(archive)

        # Extract Pareto front for this run
        run_pareto = []
        for i in range(P.size):
            if P.ind_array[i] != NULL:
                obj1 = P.ind_array[i].f[0]
                obj2 = P.ind_array[i].f[1] if nf > 1 else 0
                run_pareto.append([obj1, obj2])

        all_pareto_solutions.extend(run_pareto)
        pareto_np = np.array(run_pareto)
        if pareto_np.shape[0] > 0:
            max_obj1 = np.max(pareto_np[:, 0])
            min_obj1 = np.min(pareto_np[:, 0])
            spread_obj1 = max_obj1 - min_obj1
            max_obj2 = np.max(pareto_np[:, 1])
            min_obj2 = np.min(pareto_np[:, 1])
            spread_obj2 = max_obj2 - min_obj2
        else:
            max_obj1 = min_obj1 = spread_obj1 = 0
            max_obj2 = min_obj2 = spread_obj2 = 0

        run_time = time.time() - run_start_time
        run_times.append(run_time)

        if print_params:
            status = "⚠️" if run_time > runtime_threshold else "✓"
            print(f"{status} {len(run_pareto)} solutions, {run_time:.2f}s")

        complete_free_pop(P)
        cleanup_globals()

    total_time = time.time() - total_start_time
    avg_time_per_run = total_time / num_runs if num_runs > 0 else 0.0

    if print_params:
        print(f" Enhanced Complete{agent_info}: {len(all_pareto_solutions)} total solutions, {total_time:.2f}s total, {avg_time_per_run:.2f}s avg/run")

    return {
        'pareto_solutions': np.array(all_pareto_solutions) if all_pareto_solutions else np.array([]),
        'total_time': total_time,
        'avg_time_per_run': avg_time_per_run,
        'run_times': run_times,
        'parameters': {
            'alpha': alpha,
            'kappa': kappa,
            'L': L,
            'small_value': smallValue
        },
        'num_solutions': len(all_pareto_solutions),
        'num_runs': num_runs,
        'num_iterations': num_iterations,
        'operator_strategy': operator_strategy,
        'max_obj1': max_obj1,
        'min_obj1': min_obj1,
        'spread_obj1': spread_obj1,
        'max_obj2': max_obj2,
        'min_obj2': min_obj2,
        'spread_obj2': spread_obj2,
        'agent_id': agent_id,
        'generation': generation
    }

print(" Enhanced MOACP Implementation with Mind Evolution Ready!")

In [67]:
class MindEvolutionOptimizer:
    """Optimizer implementing Mind Evolution approach"""
    
    def __init__(self, llm_interface, config_manager, reference_hvs):
        self.llm_interface = llm_interface
        self.config_manager = config_manager
        self.reference_hvs = reference_hvs
        self.optimization_history = []
        self.best_results = {}
        self.agent_population = []
        self.generation = 0
        
    def optimize_instance_with_mind_evolution(self, instance_file, weights_file, nbitems, num_objectives, 
                                           max_generations=10):
        """
        Optimize a specific instance using Mind Evolution
        """
        # Determine instance signature
        instance_sig = f"{nbitems}_{num_objectives}"
        
        # Get reference data for this instance
        if instance_sig not in self.reference_hvs:
            print(f"❌ No reference data found for instance {instance_sig}")
            return None, None, False
        
        reference_data = self.reference_hvs[instance_sig]
        reference_hv = reference_data['hypervolume']
        target_hv = reference_hv * 1.05  # 5% improvement target
        
        print(f"\n🧠 MIND EVOLUTION OPTIMIZATION")
        print(f"Instance: {instance_file}")
        print(f"Items: {nbitems}, Objectives: {num_objectives}")
        print(f"Reference HV: {reference_hv:,.0f}")
        print(f"Target HV: {target_hv:,.0f}")
        print(f"Gap to close: {target_hv - reference_hv:,.0f}")
        
        # Extract instance features
        instance_features = self.config_manager.extract_instance_features(
            instance_file, weights_file, nbitems, num_objectives
        )
        
        print(f"Instance signature: {instance_features['signature']}")
        print(f"Instance size: {instance_features['instance_size']}")
        print(f"Subclass: {instance_features['subclass']}")
        
        # Initialize agent population
        print(f"\n🔬 Initializing agent population...")
        self.agent_population = self.llm_interface.initialize_evolutionary_population(
            instance_features, target_hv
        )
        
        print(f"Initialized {len(self.agent_population)} agents")
        
        best_result = None
        best_hv = 0
        best_config = None
        dominance_achieved = False
        
        for generation in range(max_generations):
            self.generation = generation
            print(f"\n🧬 Generation {generation + 1}/{max_generations}")
            
            # Evaluate all agents
            agent_results = []
            for i, config in enumerate(self.agent_population):
                print(f"  🤖 Agent {i+1}/{len(self.agent_population)}: α={config['alpha']}, κ={config['kappa']:.3f}, L={config['L']}")
                
                # Run with current agent configuration
                result = run_moacp_mind_evolution(
                    instance_file=instance_file,
                    weights_file=weights_file,
                    nbitems=nbitems,
                    num_objectives=num_objectives,
                    custom_params=config,
                    operator_strategy=config['operator_strategy'],
                    print_params=False,
                    runtime_threshold=config['runtime_threshold'],
                    num_runs=6,  # Reduced runs for faster evolution
                    num_iterations=80,  # Reduced iterations for faster evolution
                    agent_id=i,
                    generation=generation
                )
                
                # Calculate hypervolume
                hv = calculate_hypervolume_2d(result['pareto_solutions'])
                gap_to_target = target_hv - hv
                improvement = ((hv - reference_hv) / reference_hv) * 100
                
                print(f"    HV: {hv:,.0f} ({improvement:.2f}% improvement)")
                
                agent_results.append({
                    'agent_id': i,
                    'config': config,
                    'result': result,
                    'hv': hv,
                    'improvement': improvement,
                    'gap': gap_to_target
                })
                
                # Update best result
                if hv > best_hv:
                    best_hv = hv
                    best_result = result
                    best_config = config.copy()
                    print(f"    ✨ New best! HV: {hv:,.0f} ({improvement:.2f}% improvement)")
                
                # Check for dominance
                if hv >= target_hv:
                    excess = hv - target_hv
                    print(f"    🎉 DOMINANCE ACHIEVED! Excess={excess:,.0f}")
                    dominance_achieved = True
                    break
            
            if dominance_achieved:
                break
            
            # Sort agents by performance
            agent_results.sort(key=lambda x: x['hv'], reverse=True)
            
            # Print generation summary
            print(f"  📊 Generation {generation + 1} Summary:")
            print(f"    Best HV: {agent_results[0]['hv']:,.0f} ({agent_results[0]['improvement']:.2f}% improvement)")
            print(f"    Worst HV: {agent_results[-1]['hv']:,.0f} ({agent_results[-1]['improvement']:.2f}% improvement)")
            print(f"    Average HV: {np.mean([r['hv'] for r in agent_results]):,.0f}")
            
            # Create next generation
            if generation < max_generations - 1:
                print(f"  🔄 Creating next generation...")
                next_generation = []
                
                # Elitism: keep top 2 agents
                next_generation.append(agent_results[0]['config'].copy())
                next_generation.append(agent_results[1]['config'].copy())
                
                # Crossover: create 2 offspring from top parents
                parent1 = agent_results[0]['config']
                parent2 = agent_results[1]['config']
                
                offspring1 = self.llm_interface.evolutionary_crossover(
                    parent1, parent2, instance_features
                )
                offspring2 = self.llm_interface.evolutionary_crossover(
                    parent2, parent1, instance_features
                )
                
                next_generation.append(offspring1)
                next_generation.append(offspring2)
                
                # Mutation: mutate 1 agent
                mutation_target = agent_results[2]['config']  # Third best
                mutated = self.llm_interface.evolutionary_mutation(
                    mutation_target, instance_features, 'medium'
                )
                
                next_generation.append(mutated)
                
                self.agent_population = next_generation
                print(f"    Created {len(next_generation)} agents for next generation")
        
        # Update best configuration for this instance type
        if best_config:
            self.config_manager.update_best_config(
                instance_features['subclass'], best_config, best_hv
            )
        
        # Store best result
        if instance_features['signature'] not in self.best_results:
            self.best_results[instance_features['signature']] = {}
        
        self.best_results[instance_features['signature']] = {
            'result': best_result,
            'config': best_config,
            'hv': best_hv,
            'dominance_achieved': dominance_achieved,
            'reference_hv': reference_hv,
            'target_hv': target_hv,
            'generations': generation + 1
        }
        
        if not dominance_achieved:
            gap_remaining = target_hv - best_hv
            print(f"\n⚠️ Best effort completed. Final HV: {best_hv:,.0f}")
            print(f"Gap remaining: {gap_remaining:,.0f}")
        
        return best_result, best_config, dominance_achieved

In [68]:
print("\n=== EXECUTING MIND EVOLUTION OPTIMIZATION ===")

# First, let's ensure we have reference hypervolumes calculated
def calculate_reference_hv_for_instance(result_file):
    """Calculate reference hypervolume for a specific instance result file"""
    try:
        # Load the results
        data = np.loadtxt(result_file)
        if data.ndim == 1:
            data = data.reshape(1, -1)
        
        # Remove duplicates
        data_unique = np.unique(data, axis=0)
        
        # Extract Pareto front (maximization)
        def is_pareto_efficient(points):
            is_efficient = np.ones(points.shape[0], dtype=bool)
            for i, c in enumerate(points):
                if is_efficient[i]:
                    is_efficient[is_efficient] = np.any(points[is_efficient]>=c, axis=1)
                    is_efficient[i] = True
            return is_efficient
        
        pareto_mask = is_pareto_efficient(data_unique)
        pareto_points = data_unique[pareto_mask]
        
        # Calculate hypervolume
        hv = calculate_hypervolume_2d(pareto_points)
        
        return {
            'all_solutions': data,
            'unique_solutions': data_unique,
            'pareto_solutions': pareto_points,
            'num_pareto': len(pareto_points),
            'hypervolume': hv
        }
    except Exception as e:
        print(f"Error loading result file {result_file}: {e}")
        return None

# Calculate reference hypervolumes if not already done
reference_hvs = {}
instance_files = [
    ("250_2", "2502_Resulats.txt"),
    ("500_2", "5002_Resulats.txt"),
    ("750_2", "7502_Resulats.txt")
]

print("Calculating reference hypervolumes for all instances...")
for instance_sig, result_file in instance_files:
    print(f"Processing {result_file}...")
    ref_data = calculate_reference_hv_for_instance(result_file)
    if ref_data:
        reference_hvs[instance_sig] = ref_data
        print(f"✓ {instance_sig}: HV = {ref_data['hypervolume']:,.0f}, Pareto points = {ref_data['num_pareto']}")
    else:
        print(f"❌ Failed to process {result_file}")

print("\nReference hypervolumes calculated successfully!")

# Initialize improved LLM interface
try:
    mind_evolution_llm = MindEvolutionLLMInterface(model_path="llama3:latest", temperature=0.7)
    
    if mind_evolution_llm.connection_status == "ollama_connected":
        print("✅ Mind Evolution LLaMA-3 interface working!")
        llm_working = True
    else:
        print("⚠️ Mind Evolution LLaMA-3 interface not working, using fallback")
        llm_working = False
        
except Exception as e:
    print(f"❌ Failed to initialize Mind Evolution LLaMA-3: {e}")
    llm_working = False
    mind_evolution_llm = None

# Initialize Mind Evolution optimizer
mind_evolution_optimizer = MindEvolutionOptimizer(mind_evolution_llm, instance_config_manager, reference_hvs)
print("✅ Mind Evolution Optimizer Ready!")

# Optimize all instances
instances_to_optimize = [
    ("./multiobjectives/250.2.txt", "./multiobjectives/Weights_2obj_FQ200.txt", 250, 2),
    ("./multiobjectives/500.2.txt", "./multiobjectives/Weights_2obj_FQ200.txt", 500, 2),
    ("./multiobjectives/750.2.txt", "./multiobjectives/Weights_2obj_FQ200.txt", 750, 2)
]

results_summary = {}

for instance_file, weights_file, nbitems, num_objectives in instances_to_optimize:
    print(f"\n{'='*60}")
    print(f"MIND EVOLUTION OPTIMIZATION: {instance_file}")
    print(f"{'='*60}")
    
    # Check if files exist
    if not os.path.exists(instance_file):
        print(f"❌ Instance file not found: {instance_file}")
        continue
    
    if not os.path.exists(weights_file):
        print(f"❌ Weights file not found: {weights_file}")
        continue
    
    # Optimize this instance
    result, config, dominance_achieved = mind_evolution_optimizer.optimize_instance_with_mind_evolution(
        instance_file=instance_file,
        weights_file=weights_file,
        nbitems=nbitems,
        num_objectives=num_objectives,
        max_generations=8  # Reduced for faster execution
    )
    
    if result is None:
        print(f"❌ Failed to optimize {instance_file}")
        continue
    
    # Get instance signature
    instance_sig = f"{nbitems}_{num_objectives}"
    
    # Get reference data
    reference_data = reference_hvs[instance_sig]
    reference_hv = reference_data['hypervolume']
    
    # Calculate final hypervolume
    final_hv = calculate_hypervolume_2d(result['pareto_solutions'])
    improvement = ((final_hv - reference_hv) / reference_hv) * 100
    
    # Store results
    results_summary[instance_sig] = {
        'result': result,
        'config': config,
        'hv': final_hv,
        'reference_hv': reference_hv,
        'improvement': improvement,
        'dominance_achieved': dominance_achieved
    }
    
    # Save results
    output_file = f"mind_evolution_optimized_{instance_sig}.txt"
    np.savetxt(output_file, result['pareto_solutions'])
    print(f"Results saved to {output_file}")
    
    print(f"\n🏆 MIND EVOLUTION OPTIMIZATION COMPLETE")
    print(f"Final HV: {final_hv:,.0f}")
    print(f"Reference HV: {reference_hv:,.0f}")
    print(f"Improvement: {improvement:.2f}%")
    print(f"Dominance Achieved: {'YES' if dominance_achieved else 'NO'}")

# Print overall summary
print(f"\n{'='*60}")
print("OVERALL OPTIMIZATION SUMMARY")
print(f"{'='*60}")

for instance_sig, results in results_summary.items():
    print(f"\n{instance_sig}:")
    print(f"  Final HV: {results['hv']:,.0f}")
    print(f"  Reference HV: {results['reference_hv']:,.0f}")
    print(f"  Improvement: {results['improvement']:.2f}%")
    print(f"  Dominance Achieved: {'YES' if results['dominance_achieved'] else 'NO'}")
    if results['config']:
        print(f"  Best Config: α={results['config']['alpha']}, κ={results['config']['kappa']:.3f}, L={results['config']['L']}")


=== EXECUTING MIND EVOLUTION OPTIMIZATION ===
Calculating reference hypervolumes for all instances...
Processing 2502_Resulats.txt...
✓ 250_2: HV = 97,640,877, Pareto points = 236
Processing 5002_Resulats.txt...
✓ 500_2: HV = 405,521,824, Pareto points = 396
Processing 7502_Resulats.txt...
✓ 750_2: HV = 898,857,593, Pareto points = 412

Reference hypervolumes calculated successfully!
✅ Ollama connected with llama3:latest available
✅ Mind Evolution LLaMA-3 interface working!
✅ Mind Evolution Optimizer Ready!

MIND EVOLUTION OPTIMIZATION: ./multiobjectives/250.2.txt

🧠 MIND EVOLUTION OPTIMIZATION
Instance: ./multiobjectives/250.2.txt
Items: 250, Objectives: 2
Reference HV: 97,640,877
Target HV: 102,522,921
Gap to close: 4,882,044
Instance signature: 250_2
Instance size: small
Subclass: tight_capacity_2obj

🔬 Initializing agent population...
⚠️ JSON parsing error: Invalid \escape: line 1 column 140 (char 139)
Problematic JSON: { "alpha": 40, "kappa": 0.150, "L": 6, "operator_strategy": [

KeyboardInterrupt: 